In [1]:
%pip install PuLP

   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.4 MB 653.6 kB/s eta 0:00:26
    --------------------------------------- 0.2/16.4 MB 2.4 MB/s eta 0:00:07
   - -------------------------------------- 0.6/16.4 MB 4.3 MB/s eta 0:00:04
   --- ------------------------------------ 1.6/16.4 MB 8.4 MB/s eta 0:00:02
   ----- ---------------------------------- 2.3/16.4 MB 9.8 MB/s eta 0:00:02
   ------- -------------------------------- 3.0/16.4 MB 10.5 MB/s eta 0:00:02
   --------- ------------------------------ 3.8/16.4 MB 11.6 MB/s eta 0:00:02
   ----------- ---------------------------- 4.9/16.4 MB 13.1 MB/s eta 0:00:01
   ------------- -------------------------- 5.7/16.4 MB 13.9 MB/s eta 0:00:01
   --------------- ------------------------ 6.3/16.4 MB 13.3 MB/s eta 0:00:01
   ----------------- ---------------------- 7.0/16.4 MB 13.6 MB/s eta 0:00:01
   --

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import openpyxl
import pulp as pl

PRICE_FILE = Path("Prices.xlsx")

EMISSIONS_FACTOR = 0.05306
START_DATE = pd.to_datetime("2022-03-21").date()
END_DATE = pd.to_datetime("2022-03-27").date()

def find_col(df, candidates):
    cols_lower = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in cols_lower:
            return cols_lower[cand.lower()]
    for c in df.columns:
        c_low = c.lower()
        for cand in candidates:
            if cand.lower() in c_low:
                return c
    raise KeyError(f"Could not find any of these columns: {candidates}")

def load_data():
    price_electric = pd.read_excel(PRICE_FILE, sheet_name="PRICE_ELECTRIC")
    price_gas = pd.read_excel(PRICE_FILE, sheet_name="PRICE_GAS")

    wb = openpyxl.load_workbook(PRICE_FILE, data_only=True)
    co2_sheet = wb["PRICE_CO2"]
    co2_price = float(list(co2_sheet.values)[0][0])

    electric_date_col = find_col(price_electric, ["OPERATING_DATE", "DATE", "Trading Date"])
    electric_hour_col = find_col(price_electric, ["HOUR_ENDING", "Hour", "HE"])
    electric_price_col = find_col(price_electric, ["PRICE_ELECTRIC", "LMP", "Price"])

    gas_date_col = find_col(price_gas, ["OPERATING_DATE", "DATE", "Trading Date"])
    gas_price_col = find_col(price_gas, ["PG&E Citygate ($/MMBtu)", "PRICE_GAS", "Gas Price"])

    elec = price_electric[[electric_date_col, electric_hour_col, electric_price_col]].copy()
    elec.columns = ["OPERATING_DATE", "HOUR_ENDING", "PRICE_ELECTRIC"]

    gas = price_gas[[gas_date_col, gas_price_col]].copy()
    gas.columns = ["OPERATING_DATE", "PRICE_GAS"]

    elec["OPERATING_DATE"] = pd.to_datetime(elec["OPERATING_DATE"]).dt.date
    gas["OPERATING_DATE"] = pd.to_datetime(gas["OPERATING_DATE"]).dt.date

    elec = elec[(elec["OPERATING_DATE"] >= START_DATE) & (elec["OPERATING_DATE"] <= END_DATE)].copy()
    gas = gas[(gas["OPERATING_DATE"] >= START_DATE) & (gas["OPERATING_DATE"] <= END_DATE)].copy()

    hourly = elec.merge(gas, on="OPERATING_DATE", how="left").sort_values(["OPERATING_DATE", "HOUR_ENDING"]).reset_index(drop=True)
    return hourly, co2_price

def build_configs():
    return {
        1: {
            "name": "all-off",
            "min_mw": 0.0,
            "max_mw": 0.0,
            "vom": 0.0,
            "su_dollar": 0.0,
            "su_fuel": 0.0,
            "min_up": 0,
            "min_down": 0,
            "hr_points": {}
        },
        2: {
            "name": "single CT",
            "min_mw": 57.0,
            "max_mw": 190.0,
            "vom": 5.0,
            "su_dollar": 7250.0,
            "su_fuel": 220.0,
            "min_up": 1,
            "min_down": 1,
            "hr_points": {
                "Minimum load": 12.591,
                "60% load": 10.642,
                "80% load": 10.275,
                "100% load": 10.133
            }
        },
        3: {
            "name": "two CTs, no steam",
            "min_mw": 114.0,
            "max_mw": 380.0,
            "vom": 5.0,
            "su_dollar": 14500.0,
            "su_fuel": 440.0,
            "min_up": 1,
            "min_down": 1,
            "hr_points": {
                "Minimum load": 12.591,
                "60% load": 10.642,
                "80% load": 10.275,
                "100% load": 10.133
            }
        },
        4: {
            "name": "1x1",
            "min_mw": 150.0,
            "max_mw": 340.0,
            "vom": 2.5,
            "su_dollar": 16500.0,
            "su_fuel": 850.0,
            "min_up": 2,
            "min_down": 2,
            "hr_points": {
                "Minimum load": 7.695,
                "60% load": 7.358,
                "80% load": 7.149,
                "100% load": 7.048
            }
        },
        5: {
            "name": "2x1",
            "min_mw": 312.0,
            "max_mw": 610.0,
            "vom": 2.0,
            "su_dollar": 23750.0,
            "su_fuel": 1070.0,
            "min_up": 3,
            "min_down": 3,
            "hr_points": {
                "Minimum load": 7.121,
                "60% load": 6.999,
                "80% load": 6.839,
                "100% load": 6.762
            }
        }
    }

def solve_caiso_optimization():
    hourly, co2_price = load_data()
    configs = build_configs()

    point_labels = ["Minimum load", "60% load", "80% load", "100% load"]
    T = list(range(len(hourly)))
    C = list(configs.keys())

    point_data = {}
    point_keys = []

    for t in T:
        gas_price = float(hourly.loc[t, "PRICE_GAS"])
        for c in C:
            if c == 1:
                key = (t, c, "off")
                point_keys.append(key)
                point_data[key] = {"mw": 0.0, "var_cost": 0.0, "fuel_cost": 0.0, "co2_cost": 0.0, "vom_cost": 0.0}
                continue

            for label in point_labels:
                if label == "Minimum load":
                    mw = configs[c]["min_mw"]
                elif label == "60% load":
                    mw = 0.60 * configs[c]["max_mw"]
                elif label == "80% load":
                    mw = 0.80 * configs[c]["max_mw"]
                else:
                    mw = 1.00 * configs[c]["max_mw"]

                hr = configs[c]["hr_points"][label]
                vom = configs[c]["vom"]

                fuel_cost = hr * gas_price * mw
                co2_cost = hr * EMISSIONS_FACTOR * co2_price * mw
                vom_cost = vom * mw
                var_cost = fuel_cost + co2_cost + vom_cost

                key = (t, c, label)
                point_keys.append(key)
                point_data[key] = {
                    "mw": float(mw),
                    "var_cost": float(var_cost),
                    "fuel_cost": float(fuel_cost),
                    "co2_cost": float(co2_cost),
                    "vom_cost": float(vom_cost)
                }

    model = pl.LpProblem("CCGT_CAISO_Optimization", pl.LpMaximize)

    x = pl.LpVariable.dicts("x", [(c, t) for c in C for t in T], cat="Binary")
    y = pl.LpVariable.dicts("y", [(c, t) for c in C for t in T], cat="Binary")
    z = pl.LpVariable.dicts("z", [(c, t) for c in C for t in T], cat="Binary")
    lam = pl.LpVariable.dicts("lam", point_keys, lowBound=0, upBound=1, cat="Continuous")

    x_init = {1: 1, 2: 0, 3: 0, 4: 0, 5: 0}

    for t in T:
        model += pl.lpSum(x[(c, t)] for c in C) == 1

    for t in T:
        for c in C:
            keys_ct = [k for k in point_keys if k[0] == t and k[1] == c]
            model += pl.lpSum(lam[k] for k in keys_ct) == x[(c, t)]

    for c in C:
        for t in T:
            if t == 0:
                model += x[(c, t)] - x_init[c] == y[(c, t)] - z[(c, t)]
            else:
                model += x[(c, t)] - x[(c, t-1)] == y[(c, t)] - z[(c, t)]

    for t in T:
        if t == 0:
            model += x[(5, 0)] == 0
        else:
            model += x[(5, t)] + x[(1, t-1)] <= 1

    for c in C:
        mut = int(configs[c]["min_up"])
        if mut > 0:
            for t in T:
                if t + mut - 1 < len(T):
                    model += pl.lpSum(x[(c, tau)] for tau in range(t, t + mut)) >= mut * y[(c, t)]

    for c in C:
        mdt = int(configs[c]["min_down"])
        if mdt > 0:
            for t in T:
                if t + mdt - 1 < len(T):
                    model += pl.lpSum(1 - x[(c, tau)] for tau in range(t, t + mdt)) >= mdt * z[(c, t)]

    p_t = {}
    fuel_cost_t = {}
    co2_cost_t = {}
    vom_cost_t = {}
    var_cost_t = {}

    for t in T:
        p_t[t] = pl.lpSum(point_data[k]["mw"] * lam[k] for k in point_keys if k[0] == t)
        fuel_cost_t[t] = pl.lpSum(point_data[k]["fuel_cost"] * lam[k] for k in point_keys if k[0] == t)
        co2_cost_t[t] = pl.lpSum(point_data[k]["co2_cost"] * lam[k] for k in point_keys if k[0] == t)
        vom_cost_t[t] = pl.lpSum(point_data[k]["vom_cost"] * lam[k] for k in point_keys if k[0] == t)
        var_cost_t[t] = pl.lpSum(point_data[k]["var_cost"] * lam[k] for k in point_keys if k[0] == t)

    startup_cost_t = {}
    for t in T:
        startup_cost_t[t] = pl.lpSum(configs[c]["su_dollar"] * y[(c, t)] for c in C)

    model += pl.lpSum(
        float(hourly.loc[t, "PRICE_ELECTRIC"]) * p_t[t] - var_cost_t[t] - startup_cost_t[t]
        for t in T
    )

    solver = pl.PULP_CBC_CMD(msg=False)
    model.solve(solver)

    status = pl.LpStatus[model.status]
    if status != "Optimal":
        raise ValueError(f"Optimization did not solve to Optimal. Status = {status}")

    rows = []
    for t in T:
        active_config = max(C, key=lambda c: pl.value(x[(c, t)]))
        mw_generation = float(pl.value(p_t[t]))
        fuel_cost = float(pl.value(fuel_cost_t[t]))
        co2_cost = float(pl.value(co2_cost_t[t]))
        vom_cost = float(pl.value(vom_cost_t[t]))
        startup_cost = float(pl.value(startup_cost_t[t]))
        start_indicator = int(round(sum(pl.value(y[(c, t)]) for c in C)))

        rows.append({
            "OPERATING_DATE": hourly.loc[t, "OPERATING_DATE"],
            "HOUR_ENDING": int(hourly.loc[t, "HOUR_ENDING"]),
            "PRICE_ELECTRIC": float(hourly.loc[t, "PRICE_ELECTRIC"]),
            "PRICE_GAS": float(hourly.loc[t, "PRICE_GAS"]),
            "CONFIGURATION_ACTIVE": active_config,
            "MW_GENERATION": mw_generation,
            "FUEL_COST": fuel_cost,
            "CO2_COST": co2_cost,
            "VOM_COST": vom_cost,
            "STARTUP_COST": startup_cost,
            "START_INDICATOR": start_indicator
        })

    results = pd.DataFrame(rows)

    for col in ["PRICE_ELECTRIC", "PRICE_GAS", "MW_GENERATION", "FUEL_COST", "CO2_COST", "VOM_COST", "STARTUP_COST"]:
        results[col] = results[col].round(4)

    results[[
        "OPERATING_DATE", "HOUR_ENDING", "PRICE_ELECTRIC", "PRICE_GAS", "CONFIGURATION_ACTIVE", "MW_GENERATION"
    ]].to_csv("CCGT_CAISO.csv", index=False)

    installed_capacity_mw = 610.0
    installed_capacity_kw = 610000.0
    hours = len(results)

    total_revenue = float((results["PRICE_ELECTRIC"] * results["MW_GENERATION"]).sum())
    fuel_cost = float(results["FUEL_COST"].sum())
    co2_cost = float(results["CO2_COST"].sum())
    vom_cost = float(results["VOM_COST"].sum())
    startup_cost = float(results["STARTUP_COST"].sum())
    total_costs = fuel_cost + co2_cost + vom_cost + startup_cost

    gross_margin = total_revenue - total_costs
    gross_margin_per_kw = gross_margin / installed_capacity_kw
    capacity_factor = results["MW_GENERATION"].sum() / (installed_capacity_mw * hours)
    number_of_starts = int(results["START_INDICATOR"].sum())

    summary = pd.DataFrame({
        "Metric": [
            "Gross Margin ($/kW)",
            "Capacity Factor (%)",
            "Total Revenue ($)",
            "Total Costs ($)",
            "Fuel Costs ($)",
            "Number of Starts"
        ],
        "Value": [
            round(gross_margin_per_kw, 4),
            round(capacity_factor * 100, 4),
            round(total_revenue, 4),
            round(total_costs, 4),
            round(fuel_cost, 4),
            number_of_starts
        ]
    })

    return results, summary

if __name__ == "__main__":
    results, summary = solve_caiso_optimization()
    print(results.head(12))
    print()
    print(summary)

KeyError: "Could not find any of these columns: ['PRICE_ELECTRIC', 'LMP', 'Price']"